# Part 1 — Regression and Classification

**Dataset**: AI4I 2020 Predictive Maintenance  
**Tasks**:
1. Regression: Predict Tool Wear (continuous target)
2. Classification 1: Logistic Regression — predict Machine Failure
3. Classification 2: Random Forest — predict Machine Failure


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries loaded.')

## 1. Data Loading and EDA


In [ ]:
from src.data.loader import load_raw
from src.data.preprocessor import engineer_features

df_raw = load_raw()
df = engineer_features(df_raw)

print(f'Shape: {df.shape}')
print(f'Failure rate: {df["Machine failure"].mean()*100:.2f}%')
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Tool wear [min]'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Tool Wear Distribution')
axes[0].set_xlabel('Tool wear [min]')

vals = df['Machine failure'].value_counts()
axes[1].bar(['No Failure', 'Failure'], vals.values, color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Machine Failure Count')
for i, v in enumerate(vals.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontweight='bold')

tc = df['Type'].value_counts()
axes[2].bar(tc.index, tc.values, color=['#3498db', '#9b59b6', '#e67e22'])
axes[2].set_title('Machine Type Distribution')

plt.tight_layout()
plt.show()

In [ ]:
cols = ['Air temperature [K]', 'Process temperature [K]',
        'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
        'temp_diff', 'power', 'Machine failure']
plt.figure(figsize=(10, 8))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 2. Regression — Predicting Tool Wear

**Target**: `Tool wear [min]` — continuous, range 0-254 min  
**Model**: Linear Regression  
**Why**: Tool wear is a direct indicator of remaining useful life.


In [ ]:
REG_FEATURES = [
    'Type_encoded', 'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'temp_diff', 'power'
]

X_r = df[REG_FEATURES].values
y_r = df['Tool wear [min]'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

sc_r = StandardScaler()
X_tr = sc_r.fit_transform(X_tr)
X_te = sc_r.transform(X_te)

lr = LinearRegression()
lr.fit(X_tr, y_tr)
y_pred_r = lr.predict(X_te)

rmse = np.sqrt(mean_squared_error(y_te, y_pred_r))
mae  = mean_absolute_error(y_te, y_pred_r)
r2   = r2_score(y_te, y_pred_r)

print('=== Linear Regression — Tool Wear ===')
print(f'RMSE: {rmse:.4f}')
print(f'MAE : {mae:.4f}')
print(f'R2  : {r2:.4f}')
print()
for feat, coef in zip(REG_FEATURES, lr.coef_):
    print(f'  {feat}: {coef:+.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_te, y_pred_r, alpha=0.4, color='steelblue', s=15)
lims = [min(y_te.min(), y_pred_r.min()), max(y_te.max(), y_pred_r.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect fit')
axes[0].set_xlabel('Actual Tool Wear [min]')
axes[0].set_ylabel('Predicted Tool Wear [min]')
axes[0].set_title(f'Actual vs Predicted (R2={r2:.3f})')
axes[0].legend()

residuals = y_te - y_pred_r
axes[1].scatter(y_pred_r, residuals, alpha=0.4, color='coral', s=15)
axes[1].axhline(0, color='black', lw=2, ls='--')
axes[1].set_xlabel('Predicted Tool Wear [min]')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

## 3. Classification 1 — Logistic Regression

**Target**: `Machine failure` (binary)  
**Note**: 3.4% failure rate — severe class imbalance. Using `class_weight='balanced'`.


In [ ]:
CLF_FEATURES = [
    'Type_encoded', 'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'temp_diff', 'power', 'wear_rate'
]

X_c = df[CLF_FEATURES].values
y_c = df['Machine failure'].values

X_tc, X_vc, y_tc, y_vc = train_test_split(
    X_c, y_c, test_size=0.2, stratify=y_c, random_state=42
)

sc_c = StandardScaler()
X_tc = sc_c.fit_transform(X_tc)
X_vc = sc_c.transform(X_vc)

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_tc, y_tc)

y_pred_lr = log_reg.predict(X_vc)
y_prob_lr = log_reg.predict_proba(X_vc)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_vc, y_pred_lr, target_names=['No Failure', 'Failure']))
print(f'ROC-AUC: {roc_auc_score(y_vc, y_prob_lr):.4f}')

In [ ]:
cm = confusion_matrix(y_vc, y_pred_lr)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Failure', 'Failure'],
            yticklabels=['No Failure', 'Failure'])
plt.title('Confusion Matrix — Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 4. Classification 2 — Random Forest


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_tc, y_tc)

y_pred_rf = rf.predict(X_vc)
y_prob_rf = rf.predict_proba(X_vc)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_vc, y_pred_rf, target_names=['No Failure', 'Failure']))
print(f'ROC-AUC: {roc_auc_score(y_vc, y_prob_rf):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cm_rf = confusion_matrix(y_vc, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['No Failure', 'Failure'],
            yticklabels=['No Failure', 'Failure'])
axes[0].set_title('Confusion Matrix — Random Forest')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]
axes[1].bar(range(len(CLF_FEATURES)), importances[idx], color='steelblue')
axes[1].set_xticks(range(len(CLF_FEATURES)))
axes[1].set_xticklabels([CLF_FEATURES[i] for i in idx], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Feature Importances — Random Forest')
axes[1].set_ylabel('Importance')

plt.tight_layout()
plt.show()

## 5. Model Comparison


In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy':  [accuracy_score(y_vc, y_pred_lr), accuracy_score(y_vc, y_pred_rf)],
    'Precision': [precision_score(y_vc, y_pred_lr, zero_division=0),
                  precision_score(y_vc, y_pred_rf, zero_division=0)],
    'Recall':    [recall_score(y_vc, y_pred_lr, zero_division=0),
                  recall_score(y_vc, y_pred_rf, zero_division=0)],
    'F1':        [f1_score(y_vc, y_pred_lr, zero_division=0),
                  f1_score(y_vc, y_pred_rf, zero_division=0)],
    'ROC-AUC':   [roc_auc_score(y_vc, y_prob_lr), roc_auc_score(y_vc, y_prob_rf)],
}).set_index('Model')

print(results.round(4))

results.drop(columns=['Accuracy']).plot(kind='bar', figsize=(10, 5))
plt.title('Classification Model Comparison')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Ethical and Practical Considerations

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Class Imbalance** | 3.4% failures — model may ignore minority class | SMOTE, `class_weight='balanced'`, F2-score threshold tuning |
| **False Negatives** | Missing a failure = equipment damage or safety hazard | Optimize Recall; lower decision threshold |
| **Overfitting** | Random Forest can overfit noisy sensor data | Cross-validation, feature importance pruning |
| **Data Privacy** | Sensor data may include proprietary process info | Anonymize product IDs, aggregate at machine level |
| **Interpretability** | Black-box models hard to audit in safety-critical systems | SHAP explanations (EU AI Act compliance) |

### Conclusion
- **Regression**: Linear Regression has limited power for Tool Wear (approximately uniform distribution). R2 is low, indicating Tool Wear is largely driven by time/usage not captured by static sensor readings.
- **Classification**: Random Forest outperforms Logistic Regression for imbalanced failure detection, especially on Recall — the most critical metric for safety systems.
- **Production recommendation**: XGBoost with SMOTE and F2-score threshold optimization (implemented in `src/models/trainer.py`).
